In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from tqdm import tqdm
from functools import reduce
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import VotingRegressor

In [14]:
df=pd.read_csv("../ISI_dataset/merged_rabi_rice_reservoir.csv")
df.head()

,state_name,crop_name,apy_item_interval_start,temperature_recorded_date,state_temperature_max_val,state_temperature_min_val,state_rainfall_val,yield,FRL,Live Cap FRL,Level,Current Live Storage
0,Andhra Pradesh,rice,2000,2000-01-01,30.38,14.47,0.0,3.58525,152.296667,2.838333,266.30,6.390
1,Andhra Pradesh,rice,2000,2000-01-02,30.04,13.96,0.0,3.58525,152.296667,2.838333,266.18,6.330
2,Andhra Pradesh,rice,2000,2000-01-03,29.92,12.98,0.0,3.58525,152.296667,2.838333,266.09,6.286
3,Andhra Pradesh,rice,2000,2000-01-04,29.98,12.23,0.0,3.58525,152.296667,2.838333,266.03,6.257
4,Andhra Pradesh,rice,2000,2000-01-05,29.77,13.24,0.0,3.58525,152.296667,2.838333,265.97,6.228


In [15]:
df['temperature_recorded_date'] = pd.to_datetime(df['temperature_recorded_date'])
df['year'] = df['temperature_recorded_date'].dt.year

In [16]:
# Use only data till 2022 for training
df = df[df['year'] < 2023].copy()

# Drop unreliable states
df = df[~df['state_name'].isin(['Jharkhand', 'Uttarakhand'])]

# Group annually to match 2023 structure
df_annual = df.groupby(['state_name', 'crop_name', 'year']).agg({
    'state_rainfall_val': 'sum',
    'state_temperature_max_val': 'mean',
    'state_temperature_min_val': 'mean',
    'Live Cap FRL': 'mean',
    'FRL': 'mean',
    'Level': 'mean',
    'Current Live Storage': 'mean',
    'yield': 'mean'
}).reset_index()

In [17]:
df_annual.head()

,state_name,crop_name,year,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,yield
0,Andhra Pradesh,rice,2000,942.03,34.738197,18.885082,2.838333,152.296667,179.065178,2.636298,3.58525
1,Andhra Pradesh,rice,2001,918.85,35.137890,18.995397,2.838333,152.296667,171.736301,1.820863,3.78690
2,Andhra Pradesh,rice,2002,654.57,35.292247,18.858493,2.838333,152.296667,169.014014,1.402067,3.48012
3,Andhra Pradesh,rice,2003,832.53,35.550356,19.248548,2.838333,152.296667,161.787795,0.781289,3.98085
4,Andhra Pradesh,rice,2004,786.89,34.954836,18.399536,2.838333,152.296667,164.511298,1.314224,3.95922


In [18]:
# One-hot encode 'state_name'
df_encoded = pd.get_dummies(df_annual, columns=['state_name'])

# Define features: original + one-hot encoded state columns
state_columns = [col for col in df_encoded.columns if col.startswith('state_name_')]

In [19]:
# Define features and target
features = ['state_rainfall_val', 'state_temperature_max_val', 'state_temperature_min_val', 'Live Cap FRL', 'FRL','Level','Current Live Storage']+ state_columns

# Split manually by year
train_df = df_encoded[df_encoded['year'] <= 2020]
test_df = df_encoded[df_encoded['year'].between(2021, 2022)]

In [20]:
X_train = train_df[features]
y_train = train_df['yield']
X_test = test_df[features]
y_test = test_df['yield']

In [21]:
# Models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR()
}

# Results container
results = []

# Loop through models
for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    results.append({
        'Model': name,
        'Train R²': round(train_r2, 4),
        'Test R²': round(test_r2, 4),
        'Train RMSE': round(train_rmse, 2),
        'Test RMSE': round(test_rmse, 2)
    })

# Display results
results_df = pd.DataFrame(results)
print(results_df.sort_values(by='Test R²', ascending=False))

                      Model  Train R²  Test R²  Train RMSE  Test RMSE
4  Support Vector Regressor    0.7484   0.8556        0.40       0.31
2                   XGBoost    1.0000   0.8545        0.00       0.31
0         Linear Regression    0.8539   0.8473        0.30       0.32
1             Random Forest    0.9730   0.8287        0.13       0.33
3         Gradient Boosting    0.9983   0.7792        0.03       0.38


In [ ]:
# Initialize individual models
lr = LinearRegression()
svr = SVR()
xgb = XGBRegressor(random_state=42)

# Ensemble model
ensemble = VotingRegressor(estimators=[
    ('lr', lr),
    ('svr', svr),
    ('xgb', xgb)
])

# Fit ensemble
ensemble.fit(X_train, y_train)

# Predict
train_pred = ensemble.predict(X_train)
test_pred = ensemble.predict(X_test)

# Evaluate
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("📊 Ensemble Performance:")
print(f"Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")


📊 Ensemble Performance:
Train R²: 0.9235, Test R²: 0.8742
Train RMSE: 0.2189, Test RMSE: 0.2866


In [ ]:
# --- 1. Define function to forecast any single feature using Prophet ---
def forecast_feature_prophet(df, feature_name):
    forecast_data = []

    for (state, crop), group in tqdm(df.groupby(['state_name', 'crop_name'])):
        yearly_data = group.groupby('year')[feature_name].mean().reset_index()

        if yearly_data.shape[0] < 4:
            continue

        prophet_df = yearly_data.rename(columns={'year': 'ds', feature_name: 'y'})
        prophet_df['ds'] = pd.to_datetime(prophet_df['ds'], format='%Y')

        try:
            model = Prophet()
            model.fit(prophet_df)

            future = pd.DataFrame({'ds': [pd.to_datetime('2023')]})
            forecast = model.predict(future)
            yhat = forecast['yhat'].values[0]

            forecast_data.append({
                'state_name': state,
                'crop_name': crop,
                feature_name: yhat
            })
        except:
            continue

    return pd.DataFrame(forecast_data)

# --- 2. Forecast each feature separately ---
df_rain = forecast_feature_prophet(df, 'state_rainfall_val')
df_temp_max = forecast_feature_prophet(df, 'state_temperature_max_val')
df_temp_min = forecast_feature_prophet(df, 'state_temperature_min_val')
df_livecap = forecast_feature_prophet(df, 'Live Cap FRL')
df_frl = forecast_feature_prophet(df, 'FRL')
df_level = forecast_feature_prophet(df, 'Level')
df_cls = forecast_feature_prophet(df, 'Current Live Storage')

# --- 3. Merge all forecasted dataframes ---
from functools import reduce
dfs = [df_rain, df_temp_max, df_temp_min, df_livecap, df_frl, df_level, df_cls]
df_2023 = reduce(lambda left, right: pd.merge(left, right, on=['state_name', 'crop_name'], how='outer'), dfs)

# --- 4. One-hot encode state_name ---
df_2023_encoded = df_2023.copy()  # Keep original columns
state_names = df_2023_encoded[['state_name', 'crop_name']]  # Keep for merging later

df_2023_encoded = pd.get_dummies(df_2023_encoded, columns=['state_name'])
df_2023_encoded = pd.concat([state_names, df_2023_encoded.drop(columns=['crop_name'])], axis=1)


  0%|          | 0/3 [00:00<?, ?it/s]17:07:26 - cmdstanpy - INFO - Chain [1] start processing
17:07:27 - cmdstanpy - INFO - Chain [1] done processing
 33%|███▎      | 1/3 [00:02<00:04,  2.01s/it]17:07:27 - cmdstanpy - INFO - Chain [1] start processing
17:07:28 - cmdstanpy - INFO - Chain [1] done processing
 67%|██████▋   | 2/3 [00:02<00:01,  1.01s/it]17:07:28 - cmdstanpy - INFO - Chain [1] start processing
17:07:28 - cmdstanpy - INFO - Chain [1] done processing
  0%|          | 0/3 [00:00<?, ?it/s]17:07:28 - cmdstanpy - INFO - Chain [1] start processing
17:07:28 - cmdstanpy - INFO - Chain [1] done processing
 33%|███▎      | 1/3 [00:00<00:01,  1.99it/s]17:07:28 - cmdstanpy - INFO - Chain [1] start processing
17:07:29 - cmdstanpy - INFO - Chain [1] done processing
 67%|██████▋   | 2/3 [00:00<00:00,  2.23it/s]17:07:29 - cmdstanpy - INFO - Chain [1] start processing
17:07:29 - cmdstanpy - INFO - Chain [1] done processing
  0%|          | 0/3 [00:00<?, ?it/s]17:07:29 - cmdstanpy - INFO - C

In [24]:
df_2023_encoded.head()

,state_name,crop_name,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,state_name_Andhra Pradesh,state_name_Karnataka,state_name_Telangana
0,Andhra Pradesh,rice,2.790352,34.392923,19.347568,2.179976,198.326146,142.472016,0.996292,True,False,False
1,Karnataka,rice,3.444745,33.812358,17.518092,1.539062,597.731875,586.334229,0.790733,False,True,False
2,Telangana,rice,3.018702,33.997959,20.301551,2.109209,347.315496,368.086577,0.816613,False,False,True


In [25]:
X_2023 = df_2023_encoded[features]

# Use your trained ensemble model
y_2023_pred = ensemble.predict(X_2023)

# Add prediction to the dataframe
df_2023_encoded['predicted_yield'] = y_2023_pred

# Select output
output_2023 = df_2023_encoded[['crop_name'] + [col for col in df_2023_encoded.columns if col.startswith('state_name_')] + ['predicted_yield']]


In [30]:
# Convert dummy columns back to state_name
state_names = df_2023_encoded[[col for col in df_2023_encoded.columns if col.startswith('state_name_')]].idxmax(axis=1)
state_names = state_names.str.replace('state_name_', '')

# Final output
final_2023_yield = pd.DataFrame({
    'state_name': state_names,
    'crop_name': df_2023_encoded['crop_name'],
    'predicted_yield_2023': df_2023_encoded['predicted_yield']
})

print(final_2023_yield)
final_2023_yield.to_csv("../yield_prediction.csv", index=False)

       state_name crop_name  predicted_yield_2023
0  Andhra Pradesh      rice              3.761990
1       Karnataka      rice              2.319948
2       Telangana      rice              3.036549


In [27]:
# Step 1: Get actual yields from 2019 to 2022
df_recent = df_annual[df_annual['year'].between(2019, 2022)].copy()

# Pivot to get each year's yield as a column
yield_table = df_recent.pivot_table(
    index=['state_name', 'crop_name'],
    columns='year',
    values='yield'
).reset_index()

# Rename columns for clarity
yield_table = yield_table.rename(columns={
    2019: 'yield_2019',
    2020: 'yield_2020',
    2021: 'yield_2021',
    2022: 'yield_2022'
})

# Step 2: Prepare 2023 predicted yield
df_2023_yield = df_2023_encoded[['state_name', 'crop_name', 'predicted_yield']].copy()
df_2023_yield = df_2023_yield.rename(columns={'predicted_yield': 'yield_2023'})

# Step 3: Merge the 2023 predicted yield into the table
final_yield_table = pd.merge(yield_table, df_2023_yield, on=['state_name', 'crop_name'], how='left')

# Display final table
print(final_yield_table)


       state_name crop_name  yield_2019  yield_2020  yield_2021  yield_2022  \
0  Andhra Pradesh      rice     4.57842     4.43686     4.37868     4.75349   
1       Karnataka      rice     2.75502     2.24562     2.55879     2.66302   
2       Telangana      rice     3.72253     3.89254     3.57221     3.44465   

   yield_2023  
0    3.761990  
1    2.319948  
2    3.036549  
